# **Deep Learning : Dinosaur Challenge EDA**

**Team members**: Evgenia Dretaki

**Course**: Artificial Intelligence, Thomas More

**Introduction**:

This notebook performs exploratory data analysis (EDA) and data cleaning for the dinosaur image classification challenge. The dataset includes many images per dinosaur species (Ankylosaurus, Diplodocus, Parasaurolophus, Stegosaurus, Tyrannosaurus Rex, Triceratops, Velociraptor) in the training set and test images identified by IDs. We analyze class distribution, visualize sample images, check image properties (dimensions, color distributions), and clean the dataset by identifying issues (e.g., missing files, invalid formats). The cleaned data is prepared using `ImageDataGenerator` for training, validation, and test sets, with metadata saved for the modeling notebook.

**Objectives:**

- Analyze the number of images per class to identify imbalances.
- Visualize sample images to understand visual diversity.
- Examine image properties (dimensions, color distributions) to guide preprocessing.
- Check for data issues (missing files, inconsistent formats) and clean the dataset.
- Prepare data generators for training, validation, and test sets.
- Save metadata (e.g., class indices, test image paths) for modeling.

**Import required libraries for data analysis, visualization, and preprocessing**


In [1]:
pip install --quiet --no-cache-dir -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [32 lines of output]
      Traceback (most recent call last):
        File "c:\Users\30698\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "c:\Users\30698\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\Users\30698\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 137, in get_requires_for_build_wheel
          backend = _build_backend()
        File "c:\Users\30698\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_ven

In [9]:
import tensorflow as tf  # TensorFlow for deep learning and data generators
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # For loading and augmenting images
import numpy as np  # For numerical operations and array handling
import pandas as pd  # For handling CSV files and dataframes
import matplotlib.pyplot as plt  # For plotting visualizations
import seaborn as sns  # For enhanced visualizations like bar plots
import os  # For file and directory operations
import cv2  # For image processing (e.g., reading image dimensions)
from PIL import Image  # For verifying image integrity
import json  # For saving metadata to a JSON file

ModuleNotFoundError: No module named 'tensorflow'

**Define constants to ensure consistency across scripts**

In [10]:
IMG_SIZE = (224, 224)  # Target image size for resizing (224x224 for MobileNetV2)
BATCH_SIZE = 32  # Batch size for data generators to process images in batches
NUM_CLASSES = 7  # Number of dinosaur species to classify


**Define class labels mapping integer labels to dinosaur species**
**This matches the challenge's label definitions**

In [11]:
class_labels = {
    0: 'Ankylosaurus',
    1: 'Diplodocus',
    2: 'Parasaurolophus',
    3: 'Stegosaurus',
    4: 'Tyrannosaurus Rex',
    5: 'Triceratops',
    6: 'Velociraptor'
}


**Define directory paths for training and test data**
**Adjust these paths to match your dataset's location**

In [2]:
train_dir = 'Data\train\train'  # Directory with subfolders for each class (e.g., train/Ankylosaurus)
test_dir = 'Data\test\test'  # Directory with test images named by ID (e.g., test/1.jpg)

## --- Exploratory Data Analysis (EDA) ---

**Count images per class and check for missing or invalid files**

In [3]:
class_counts = {}  # Dictionary to store the number of images per class
missing_files = []  # List to track missing directories or invalid images
for class_name in class_labels.values():  # Iterate through each dinosaur species
    class_path = os.path.join(train_dir, class_name)  # Path to class subdirectory
    if os.path.exists(class_path):  # Check if the class directory exists
        # Filter files with valid image extensions (case-insensitive)
        files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        class_counts[class_name] = len(files)  # Store the number of images
        for f in files:  # Check each image for validity
            try:
                img = Image.open(os.path.join(class_path, f))  # Open the image
                img.verify()  # Verify image integrity (e.g., not corrupted)
            except:
                # If image is invalid, add to missing_files list
                missing_files.append(os.path.join(class_path, f))
    else:
        # If class directory is missing, set count to 0 and note the issue
        class_counts[class_name] = 0
        missing_files.append(f'Missing directory: {class_path}')

NameError: name 'class_labels' is not defined

**Visualize class distribution with a bar plot**

In [14]:
plt.figure(figsize=(10, 6))  # Set figure size for readability
sns.barplot(x=list(class_counts.values()), y=list(class_counts.keys()))  # Create bar plot
plt.title('Distribution of Images per Dinosaur Species')  # Add title
plt.xlabel('Number of Images')  # Label x-axis
plt.ylabel('Species')  # Label y-axis
plt.savefig('class_distribution.png')  # Save plot to file
plt.close()  # Close plot to free memory

NameError: name 'plt' is not defined

**Print class counts and any data issues**

In [2]:
print('Number of images per class:')  # Display header
for class_name, count in class_counts.items():  # Iterate through class counts
    print(f'{class_name}: {count} images')  # Print number of images per class
print(f'Total training images: {sum(class_counts.values())}')  # Print total images
if missing_files:  # Check if there are any issues
    print('Issues found:')  # Display header for issues
    for issue in missing_files:  # List each issue
        print(f'- {issue}')
else:
    print('No missing or invalid files detected.')  # Confirm data is clean

Number of images per class:


NameError: name 'class_counts' is not defined

**Check test set for missing files**

In [ ]:
test_df = pd.read_csv('Data/test.csv')  # Load test.csv with test image IDs
# Create list of expected test image paths (e.g., test/1.jpg)
test_image_paths = [os.path.join(test_dir, f'{id}.jpg') for id in test_df['id']]
# Identify any missing test files
missing_test_files = [path for path in test_image_paths if not os.path.exists(path)]
print(f'Total test images: {len(test_df)}')  # Print number of test images
if missing_test_files:  # Check for missing test files
    print('Missing test files:')  # Display header
    for path in missing_test_files:  # List each missing file
        print(f'- {path}')
else:
    print('All test files found.')  # Confirm all test files are present

NameError: name 'pd' is not defined

**Visualize sample images from each class**

In [ ]:
plt.figure(figsize=(15, 5))  # Set figure size for multiple images
for i, class_name in enumerate(class_labels.values()):  # Iterate through classes
    class_path = os.path.join(train_dir, class_name)  # Path to class directory
    if os.path.exists(class_path) and os.listdir(class_path):  # Check if directory exists
        # Get first valid image file
        img_file = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))][0]
        img_path = os.path.join(class_path, img_file)  # Full path to image
        img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)  # Load and resize image
        plt.subplot(1, NUM_CLASSES, i+1)  # Create subplot for each class
        plt.imshow(img)  # Display image
        plt.title(class_name)  # Add class name as title
        plt.axis('off')  # Hide axes for clarity
plt.savefig('sample_images.png')  # Save sample images to file
plt.close()  # Close plot to free memory

NameError: name 'plt' is not defined

**Analyze image properties (dimensions and color distribution)**

In [17]:
image_dims = []  # List to store image dimensions (height, width)
color_means = []  # List to store mean RGB values per image
for class_name in class_labels.values():  # Iterate through classes
    class_path = os.path.join(train_dir, class_name)  # Path to class directory
    if os.path.exists(class_path):  # Check if directory exists
        # Sample up to 5 images per class to avoid excessive processing
        files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:5]
        for img_file in files:  # Process each sampled image
            img_path = os.path.join(class_path, img_file)  # Full path to image
            img = cv2.imread(img_path)  # Read image with OpenCV
            if img is not None:  # Check if image was loaded successfully
                image_dims.append(img.shape[:2])  # Store height and width
                color_means.append(img.mean(axis=(0, 1)))  # Store mean RGB values

NameError: name 'os' is not defined

**Plot image dimensions as a scatter plot**

In [18]:
image_dims = np.array(image_dims)  # Convert to numpy array for plotting
plt.figure(figsize=(8, 6))  # Set figure size
plt.scatter(image_dims[:, 1], image_dims[:, 0], alpha=0.5)  # Plot width vs height
plt.title('Image Dimensions (Width vs Height)')  # Add title
plt.xlabel('Width')  # Label x-axis
plt.ylabel('Height')  # Label y-axis
plt.savefig('image_dimensions.png')  # Save plot
plt.close()  # Close plot

NameError: name 'np' is not defined

**Plot RGB color distribution as histograms**

In [19]:
color_means = np.array(color_means)  # Convert to numpy array
plt.figure(figsize=(10, 6))  # Set figure size
plt.hist(color_means[:, 0], bins=30, alpha=0.5, label='Red', color='r')  # Red channel histogram
plt.hist(color_means[:, 1], bins=30, alpha=0.5, label='Green', color='g')  # Green channel histogram
plt.hist(color_means[:, 2], bins=30, alpha=0.5, label='Blue', color='b')  # Blue channel histogram
plt.title('RGB Color Distribution Across Images')  # Add title
plt.xlabel('Mean Pixel Value')  # Label x-axis
plt.ylabel('Frequency')  # Label y-axis
plt.legend()  # Show legend
plt.savefig('color_distribution.png')  # Save plot
plt.close()  # Close plot

NameError: name 'np' is not defined

## --- Data Cleaning and Preparation ---

**Remove invalid files (if any)**

In [20]:
for invalid_file in missing_files:  # Iterate through invalid files
    if os.path.isfile(invalid_file):  # Check if the path is a file
        print(f'Removing invalid file: {invalid_file}')  # Log removal
        os.remove(invalid_file)  # Delete the invalid file

**Create training data generator with augmentation**

In [21]:
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Rescale pixel values to [0,1] for MobileNetV2
    rotation_range=20,  # Randomly rotate images up to 20 degrees
    width_shift_range=0.2,  # Randomly shift images horizontally by 20%
    height_shift_range=0.2,  # Randomly shift images vertically by 20%
    shear_range=0.2,  # Apply shear transformation
    zoom_range=0.2,  # Randomly zoom in/out by 20%
    horizontal_flip=True,  # Randomly flip images horizontally
    fill_mode='nearest',  # Fill new pixels with nearest value
    validation_split=0.2  # Reserve 20% of data for validation
)

NameError: name 'ImageDataGenerator' is not defined

**Create test data generator (only rescaling)**

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)  # Rescale test images to [0,1]

**Load training data generator**

In [22]:
train_generator = train_datagen.flow_from_directory(
    train_dir,  # Training directory
    target_size=IMG_SIZE,  # Resize images to 224x224
    batch_size=BATCH_SIZE,  # Process 32 images per batch
    class_mode='categorical',  # Use one-hot encoded labels for multi-class
    subset='training'  # Use training subset (80% of data)
)

NameError: name 'train_datagen' is not defined

**Load validation data generator**

In [23]:
validation_generator = train_datagen.flow_from_directory(
    train_dir,  # Training directory
    target_size=IMG_SIZE,  # Resize images to 224x224
    batch_size=BATCH_SIZE,  # Process 32 images per batch
    class_mode='categorical',  # Use one-hot encoded labels
    subset='validation'  # Use validation subset (20% of data)
)

NameError: name 'train_datagen' is not defined

**Prepare test data generator**

In [24]:
test_df = pd.read_csv('test.csv')  # Reload test.csv
test_image_paths = [os.path.join(test_dir, f'{id}.jpg') for id in test_df['id']]  # Create test image paths
test_df['filename'] = test_image_paths  # Add file paths to dataframe

NameError: name 'pd' is not defined

**Filter out missing test files**

In [ ]:
test_df = test_df[test_df['filename'].apply(os.path.exists)]  # Keep only existing files
if len(test_df) < len(test_image_paths):  # Check if any files were filtered
    print(f'Warning: {len(test_image_paths) - len(test_df)} test images missing. Proceeding with available images.')

**Create test data generator**

In [25]:

test_generator = test_datagen.flow_from_dataframe(
    test_df,  # Dataframe with test image paths
    x_col='filename',  # Column with file paths
    y_col=None,  # No labels for test data
    target_size=IMG_SIZE,  # Resize images to 224x224
    batch_size=BATCH_SIZE,  # Process 32 images per batch
    class_mode=None,  # No labels, only images
    shuffle=False  # Preserve order for submission
)

NameError: name 'test_datagen' is not defined

**Save metadata for modeling script**

In [ ]:

metadata = {
    'class_indices': train_generator.class_indices,  # Mapping of class names to indices
    'test_ids': test_df['id'].tolist(),  # List of test image IDs
    'test_filenames': test_df['filename'].tolist(),  # List of test image paths
    'class_counts': class_counts  # Number of images per class
}
with open('data_metadata.json', 'w') as f:  # Save metadata to JSON file
    json.dump(metadata, f)
print('Metadata saved to data_metadata.json')  # Confirm metadata is saved

NameError: name 'class_labels' is not defined